# Paper results: every condition in paper.sh

One row per condition, metrics read from each run's `preds/meta.json` — no re-encoding, runs
on CPU in seconds. The condition list is **hard-coded**: the union of the 2026-08-29 run
(the classic/mse/cosent grid, still valid — nothing that trains them changed since) and the
2026-08-31 run (the infonce/siglip families and both ablations). Default metric everywhere:
**Recall@20**; text panels on the left, image on the right.

Human-written queries (`human_study/`, `preds_human/`) are scored for every test-row model
and drawn as a third query kind (`human`) beside original/rephrased in sections 3 to 5, taken
from the rephrased-trained model of each slot.

Sections: 1 health · 2 easy × V ablation · 3 protocol comparison · 4 win rate vs hard negative · 5 full table

In [ ]:
# @claude, when editing the notebook, please actually run it
import json
import os
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import pandas as pd
import seaborn as sns

from utils.paper_analysis import discover_runs, health_check

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

NOTE = "paper"
MODELS_ROOT = "models"
K_MAIN = 20                      # the paper's headline cutoff (full table, sorting)
# Text saturates -- at recall@20 most conditions sit above 0.9, and by 2026-09-03 the top
# ten validation cells were within 0.014 of each other at recall@10 as well -- so the text
# panels and the text hparam selection use recall@5 (top-ten spread 0.019). Image is far
# from saturation and stays at @20.
K_TEXT = 5
K_IMAGE = 20
KS = (1, 5, 10, 20, 100)


def k_for(modality):
    return K_TEXT if modality == "text" else K_IMAGE
FIG_DIR = "paper/figs"
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Conditions and health

Columns: modality, style, query_kind, V, easy. `V`/`easy` are None/20 for styles that never
see the measured distance. The merge is on all five keys, so the easy-ablation variants of
one (style, V) stay distinct.

In [ ]:
# 2026-08-29 run -- the pair/margin grid; unchanged by anything since.
CONDITIONS_0829 = [
    ("text", "untrained",        "original", None, 20),
    ("text", "untrained",        "synthetic", None, 20),
    ("text", "untrained",        "rephrased", None, 20),
    ("text", "baseline-triplet", "original", None, 20),
    ("text", "baseline-triplet", "synthetic", None, 20),
    ("text", "baseline-triplet", "rephrased", None, 20),
    ("text", "cosent",           "original", None, 20),
    ("text", "cosent",           "synthetic", None, 20),
    ("text", "cosent",           "rephrased", None, 20),
    ("text", "classic-mse",      "original", 40, 20),
    ("text", "classic-mse",      "synthetic", 40, 20),
    ("text", "classic-mse",      "rephrased", 40, 20),
    ("text", "ours-mse",         "original", 40, 20),
    ("text", "ours-mse",         "synthetic", 40, 20),
    ("text", "ours-mse",         "rephrased", 40, 20),
    ("text", "ours-mse-batched", "original", 40, 20),
    ("text", "ours-mse-batched", "synthetic", 40, 20),
    ("text", "ours-mse-batched", "rephrased", 40, 20),
    ("text", "ours-mse",         "synthetic", 20, 20),
    ("text", "ours-mse",         "synthetic", 60, 20),
    ("text", "ours-mse-batched", "synthetic", 20, 20),
    ("text", "ours-mse-batched", "synthetic", 60, 20),
    ("multimodal", "untrained",        "synthetic", None, 20),
    ("multimodal", "untrained",        "rephrased", None, 20),
    ("multimodal", "baseline-triplet", "synthetic", None, 20),
    ("multimodal", "baseline-triplet", "rephrased", None, 20),
    ("multimodal", "cosent",           "synthetic", None, 20),
    ("multimodal", "cosent",           "rephrased", None, 20),
    ("multimodal", "classic-mse",      "synthetic", 40, 20),
    ("multimodal", "classic-mse",      "rephrased", 40, 20),
    ("multimodal", "ours-mse",         "synthetic", 40, 20),
    ("multimodal", "ours-mse",         "rephrased", 40, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 20),
    ("multimodal", "ours-mse-batched", "rephrased", 40, 20),
    ("multimodal", "ours-mse",         "synthetic", 20, 20),
    ("multimodal", "ours-mse",         "synthetic", 60, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 20, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 60, 20),
]

# 2026-08-31 run -- infonce/siglip families, V ablation, easy ablation.
CONDITIONS_0831 = [
    ("text", "infonce",          "original", None, 20),
    ("text", "infonce",          "synthetic", None, 20),
    ("text", "infonce",          "rephrased", None, 20),
    ("text", "infonce-mined",    "original", None, 20),
    ("text", "infonce-mined",    "synthetic", None, 20),
    ("text", "infonce-mined",    "rephrased", None, 20),
    ("text", "siglip-mined",     "original", None, 20),
    ("text", "siglip-mined",     "synthetic", None, 20),
    ("text", "siglip-mined",     "rephrased", None, 20),
    ("text", "ours-infonce",     "original", 40, 20),
    ("text", "ours-infonce",     "synthetic", 40, 20),
    ("text", "ours-infonce",     "rephrased", 40, 20),
    ("text", "ours-siglip",      "original", 40, 20),
    ("text", "ours-siglip",      "synthetic", 40, 20),
    ("text", "ours-siglip",      "rephrased", 40, 20),
    ("text", "ours-infonce",     "synthetic", 20, 20),
    ("text", "ours-infonce",     "synthetic", 60, 20),
    ("text", "ours-siglip",      "synthetic", 20, 20),
    ("text", "ours-siglip",      "synthetic", 60, 20),
    ("text", "classic-mse",      "synthetic", 40, 30),
    ("text", "classic-mse",      "synthetic", 40, 40),
    ("text", "ours-mse",         "synthetic", 40, 30),
    ("text", "ours-mse",         "synthetic", 40, 40),
    ("text", "ours-mse-batched", "synthetic", 40, 30),
    ("text", "ours-mse-batched", "synthetic", 40, 40),
    ("text", "ours-siglip",      "synthetic", 40, 30),
    ("text", "ours-siglip",      "synthetic", 40, 40),
    ("multimodal", "infonce",          "synthetic", None, 20),
    ("multimodal", "infonce",          "rephrased", None, 20),
    ("multimodal", "infonce-mined",    "synthetic", None, 20),
    ("multimodal", "infonce-mined",    "rephrased", None, 20),
    ("multimodal", "siglip-mined",     "synthetic", None, 20),
    ("multimodal", "siglip-mined",     "rephrased", None, 20),
    ("multimodal", "ours-infonce",     "synthetic", 40, 20),
    ("multimodal", "ours-infonce",     "rephrased", 40, 20),
    ("multimodal", "ours-siglip",      "synthetic", 40, 20),
    ("multimodal", "ours-siglip",      "rephrased", 40, 20),
    ("multimodal", "ours-infonce",     "synthetic", 20, 20),
    ("multimodal", "ours-infonce",     "synthetic", 60, 20),
    ("multimodal", "ours-siglip",      "synthetic", 20, 20),
    ("multimodal", "ours-siglip",      "synthetic", 60, 20),
    ("multimodal", "classic-mse",      "synthetic", 40, 30),
    ("multimodal", "classic-mse",      "synthetic", 40, 40),
    ("multimodal", "ours-mse",         "synthetic", 40, 30),
    ("multimodal", "ours-mse",         "synthetic", 40, 40),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 30),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 40),
    ("multimodal", "ours-siglip",      "synthetic", 40, 30),
    ("multimodal", "ours-siglip",      "synthetic", 40, 40),
]

# ours-infonce-margin run (queued 2026-08-31): mined infonce + distance-scheduled logit
# margins. Main grid V=40, V ablation 10/20/60 on synthetic. Rows show as unhealthy
# ("no preds") until the run completes.
CONDITIONS_MARGIN = [
    ("text",       "ours-infonce-margin", "original",  40, 20),
    ("text",       "ours-infonce-margin", "synthetic", 40, 20),
    ("text",       "ours-infonce-margin", "rephrased", 40, 20),
    ("text",       "ours-infonce-margin", "synthetic", 10, 20),
    ("text",       "ours-infonce-margin", "synthetic", 20, 20),
    ("text",       "ours-infonce-margin", "synthetic", 60, 20),
    ("multimodal", "ours-infonce-margin", "synthetic", 40, 20),
    ("multimodal", "ours-infonce-margin", "rephrased", 40, 20),
    ("multimodal", "ours-infonce-margin", "synthetic", 10, 20),
    ("multimodal", "ours-infonce-margin", "synthetic", 20, 20),
    ("multimodal", "ours-infonce-margin", "synthetic", 60, 20),
]

# mse-mined: ours-mse-batched's binary control, at ours-mse-batched's tuned hparams so the
# two see identical batches and differ in one target cell only.
CONDITIONS_MSE_MINED = [
    ("text",       "mse-mined", "original",  40, 10),
    ("text",       "mse-mined", "synthetic", 40, 10),
    ("text",       "mse-mined", "rephrased", 40, 10),
    ("multimodal", "mse-mined", "synthetic", 80, 10),
    ("multimodal", "mse-mined", "rephrased", 80, 10),
]

conditions = pd.DataFrame(CONDITIONS_0829 + CONDITIONS_0831 + CONDITIONS_MARGIN + CONDITIONS_MSE_MINED,
                          columns=["modality", "style", "query_kind", "V", "easy"])

# The lists above are history: what was run and when. The main grid paper.sh trains *now* is
# read from the script itself, so a retuned row (new V or easy) reaches `results` without
# anyone editing this cell. Duplicates between the two sources collapse to one condition.
from utils.paper_analysis import parse_conditions


def easy_of(extra):
    for token in (extra or "").split(","):
        if token.startswith("easy="):
            return int(token[len("easy="):])
    return 20


current = parse_conditions("paper.sh")
current = current[~current["extra"].fillna("").str.contains("split=val")]
current = pd.DataFrame({"modality": current["modality"], "style": current["style"],
                        "query_kind": current["query_kind"], "V": current["V"],
                        "easy": current["extra"].map(easy_of), "negs": current["negs"],
                        "mining": current["mining"], "seed": current["seed"],
                        "rephrase": current["rephrase"]})
# The historical lists predate retrieval-mined negatives and repeated trials; every row
# there is labeled and is the seed-42 trial.
conditions["negs"] = "labeled"
conditions["mining"] = ""
conditions["seed"] = 42
conditions["rephrase"] = ""
conditions = pd.concat([conditions, current], ignore_index=True)
conditions["V"] = conditions["V"].astype("float64")
conditions = conditions.drop_duplicates().reset_index(drop=True)

runs = discover_runs(MODELS_ROOT, note=NOTE)
runs["easy"] = runs["easy"].fillna(20).astype(int)
matched = conditions.merge(runs, on=["modality", "style", "query_kind", "V", "easy", "negs", "mining", "seed", "rephrase"], how="left")
for column in ("has_preds", "has_human_preds"):
    matched[column] = matched[column].fillna(False).astype(bool)

health = health_check(matched)
print(f"{len(conditions)} conditions | {int(health['healthy'].sum())} usable | "
      f"{int((~health['healthy']).sum())} not")
display(health[~health["healthy"]][["modality", "style", "query_kind", "V", "n_queries", "problem"]])
usable = matched[health["healthy"].to_numpy()].reset_index(drop=True)

# Human-query preds ride along with every test row (paper.sh Phase 2 writes preds_human/).
# The human set is small (131 text / 54 image queries on 2026-09-10), so the query floor
# is lower than the test split's.
human_health = health_check(matched, min_queries=30, preds_subdir="preds_human")
print(f"{int(human_health['healthy'].sum())} conditions with usable human-query preds | "
      f"{int((~human_health['healthy']).sum())} without")
usable_human = matched[human_health["healthy"].to_numpy()].reset_index(drop=True)

In [ ]:
# In-context rephrasings (paper.sh rephrase=in-context, 2026-09-11) are the same rephrased
# main-grid rows retrained on the _rephrased-in-context dataset and scored on its test split
# and on the _human-in-context set. They show as two more query kinds, suffixed "-ic", beside
# the plain rephrased and human bars; train_kind stays "rephrased" so they share its hparams.
IC_SUFFIX = "-ic"


def scored_kind(row, preds_subdir):
    base = "human" if preds_subdir == "preds_human" else row.query_kind
    return base + IC_SUFFIX if row.rephrase == "in-context" else base


def condition_metrics(row, preds_subdir="preds"):
    """One results row. train_kind is the query kind the model trained on and query_kind the
    kind it is scored on; they differ for human queries, which every model is scored on, and
    carry the -ic suffix for in-context models."""
    meta = json.load(open(os.path.join(row.run_dir, preds_subdir, "meta.json")))
    out = {"modality": row.modality, "style": row.style,
           "query_kind": scored_kind(row, preds_subdir),
           "train_kind": row.query_kind, "rephrase": row.rephrase,
           "V": row.V, "easy": row.easy, "negs": row.negs, "mining": row.mining,
           "order": row.order, "seed": row.seed, "n_queries": meta["n_queries"]}
    for k in KS:
        out[f"recall@{k}"] = meta["metrics"][f"recall@{k}"]
    return out

results = pd.DataFrame([condition_metrics(r) for r in usable.itertuples()]
                       + [condition_metrics(r, "preds_human") for r in usable_human.itertuples()])
results["query_kind"] = pd.Categorical(results["query_kind"],
                                       ["original", "rephrased", "synthetic", "human",
                                        "rephrased" + IC_SUFFIX, "human" + IC_SUFFIX], ordered=True)
# One column that already holds each row's modality-appropriate cutoff, so the ablation
# pivots below stay a single metric column instead of one per modality.
results["recall@k_mod"] = [row[f"recall@{k_for(row['modality'])}"]
                           for _, row in results.iterrows()]
print(f"{len(results)} conditions loaded")

## 2. easy × V ablation

One heatmap per style: the easy-negative distance against the distance normalizer V, on
**synthetic queries**. These were two separate sections, each holding the other knob at its
main-grid default (V=40 / easy=20). The knobs interact, so neither 1-D slice generalises --
for text `ours-mse-batched`, V=80 is the *worst* column at easy=10 and the *best* at easy=40.
A grid shows that; two crosshairs through it cannot.

Read from `preds_val/`, not `preds/`: the grid was swept on the validation split, which is
also the right split to choose V and easy on. The test numbers in later sections are not
touched by this selection.

Only styles with all nine cells are drawn. `classic-mse` and `ours-infonce` were swept at one
point each and are listed under the figure instead.


In [ ]:
# The easy x V grid lives in preds_val/, so it is read from the run dirs rather than reused
# from `results` (which loads test preds). discover_runs is used instead of the hard-coded
# condition table above because this grid was swept after that table was written.
EASY_LEVELS = [10, 20, 40]
V_LEVELS = [20.0, 40.0, 80.0]
V3_STYLE = "infonce-ours-v3"
V3_V_LEVELS = [10.0, 20.0, 40.0, 80.0]


def val_condition(run):
    meta_path = os.path.join(run.run_dir, "preds_val", "meta.json")
    if not os.path.exists(meta_path):
        return None
    meta = json.load(open(meta_path))
    return {"modality": run.modality, "style": run.style, "query_kind": run.query_kind,
            "V": run.V, "easy": run.easy, "negs": run.negs, "mining": run.mining,
            "rephrase": run.rephrase, "order": run.order,
            "recall@k_mod": meta["metrics"][f"recall@{k_for(run.modality)}"]}


val_runs = discover_runs(MODELS_ROOT, note=NOTE)
val_runs["easy"] = val_runs["easy"].fillna(20).astype(int)
ablation = pd.DataFrame([row for row in (val_condition(r) for r in val_runs.itertuples())
                         if row is not None])
# 2026-09-09: the paper reports rephrased (and text original) only, so the sweep panels show
# the rephrased validation grids. Styles swept on synthetic alone show as incomplete here.
ABLATION_KIND = "rephrased"
ablation = ablation[ablation["query_kind"] == ABLATION_KIND]
print(f"{len(ablation)} validation-split {ABLATION_KIND} conditions loaded")


In [ ]:
grid = ablation[ablation["easy"].isin(EASY_LEVELS) & ablation["V"].isin(V_LEVELS)]

for modality in ["text", "multimodal"]:
    sub = grid[grid["modality"] == modality]
    # ours-infonce is the deprecated GradedInfoNCELoss (soft-target grading, not the
    # margin loss). It never had a real grid -- 1/9 cells, and easy=10 collides with its
    # own label check in train.py -- and its name reads too close to ours-infonce-margin,
    # so it is dropped here rather than reported as an incomplete style.
    # infonce-ours-v3 has no easy axis (see the next cell), so it is drawn there instead.
    tables = {style: sub[sub["style"] == style]
              .pivot_table(index="easy", columns="V", values="recall@k_mod")
              .reindex(index=EASY_LEVELS, columns=V_LEVELS)
              for style in sorted(sub["style"].unique())
              if style not in ("ours-infonce", V3_STYLE)}
    full = [s for s, t in tables.items() if t.notna().to_numpy().all()]
    sparse = [s for s in tables if s not in full]

    # One colour scale across the row, so cells are comparable between styles and not just
    # within one heatmap.
    lo = min(tables[s].to_numpy().min() for s in full)
    hi = max(tables[s].to_numpy().max() for s in full)
    fig, axes = plt.subplots(1, len(full), figsize=(4.0 * len(full), 3.8), squeeze=False)
    for ax, style in zip(axes[0], full):
        table = tables[style]
        values = table.to_numpy()
        im = ax.imshow(values, cmap="viridis", vmin=lo, vmax=hi, aspect="auto")
        ax.grid(False)  # the seaborn whitegrid theme would draw rules across the cells
        best = values.argmax()
        for i, easy in enumerate(EASY_LEVELS):
            for j, v in enumerate(V_LEVELS):
                value = table.loc[easy, v]
                ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=9,
                        fontweight="bold" if i * len(V_LEVELS) + j == best else "normal",
                        color="white" if value < (lo + hi) / 2 else "black")
        ax.set_xticks(range(len(V_LEVELS)), [str(int(v)) for v in V_LEVELS])
        ax.set_yticks(range(len(EASY_LEVELS)), [str(e) for e in EASY_LEVELS])
        ax.set_xlabel("V")
        ax.set_ylabel("easy-negative distance")
        ax.set_title(style, fontsize=10)
    fig.colorbar(im, ax=axes[0], fraction=0.025, label=f"Recall@{k_for(modality)} (val)")
    # The selection metric differs by modality -- Recall@5 for text, Recall@20 for image,
    # via k_for -- so name it in the title instead of leaving it to the colourbar alone.
    fig.suptitle(f"{'text' if modality == 'text' else 'image'} \u2014 easy x V "
                 "({ABLATION_KIND}, validation split)\n"
                 f"hparams selected on Recall@{k_for(modality)}; bold cell = argmax",
                 fontweight="bold", y=1.10)
    fig.savefig(os.path.join(FIG_DIR, f"easy_v_ablation_{modality}.png"), dpi=150,
                bbox_inches="tight")
    plt.show()
    if sparse:
        print(f"{modality}: incomplete grid, not drawn -- "
              + ", ".join(f"{s} ({int(tables[s].notna().to_numpy().sum())}/9)" for s in sparse))


In [ ]:
# infonce-ours-v3 has no easy axis: random products hold target mass 0 whatever easy is, and
# the label only identifies them. Its sweep is one-dimensional in V, so it is drawn as a line
# per modality with the two references scored on the same validation split: infonce-mined
# (the ungraded control, no V) and ours-infonce-margin at its selected (V=80, easy=10).
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, modality in zip(axes, ["text", "multimodal"]):
    sub = ablation[(ablation["modality"] == modality) & (ablation["negs"] == "labeled")
                   & (ablation["rephrase"] == "")]
    sweep = (sub[sub["style"] == V3_STYLE].set_index("V")["recall@k_mod"]
             .reindex(V3_V_LEVELS))
    ax.plot(V3_V_LEVELS, sweep.to_numpy(), marker="o", label=V3_STYLE)
    if sweep.notna().any():
        best_v = sweep.idxmax()
        ax.annotate(f"V={int(best_v)}: {sweep[best_v]:.3f}", (best_v, sweep[best_v]),
                    textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
    for style, sel in (("infonce-mined", sub["V"].isna()),
                       ("ours-infonce-margin", (sub["V"] == 80) & (sub["easy"] == 10))):
        ref = sub[(sub["style"] == style) & sel]
        if not ref.empty:
            ax.axhline(ref["recall@k_mod"].iloc[0], linestyle="--", linewidth=1, label=style)
    ax.set_xscale("log", base=2)
    ax.set_xticks(V3_V_LEVELS, [str(int(v)) for v in V3_V_LEVELS])
    ax.set_xlabel("V")
    ax.set_ylabel(f"Recall@{k_for(modality)} (val)")
    ax.set_title("text" if modality == "text" else "image", fontsize=10)
    ax.legend(fontsize=8)
    missing = [int(v) for v in V3_V_LEVELS if pd.isna(sweep.get(v))]
    if missing:
        ax.text(0.02, 0.02, f"missing V: {missing}", transform=ax.transAxes,
                fontsize=8, color="0.3")
fig.suptitle(f"{V3_STYLE} \u2014 V sweep ({ABLATION_KIND}, validation split); "
             f"V selected on Recall@{K_TEXT} text / Recall@{K_IMAGE} image",
             fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "v_sweep_infonce_ours_v3.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 50/50 mixed negatives (paper.sh "50/50 mixed negatives", 2026-09-11): infonce-ours-v3 on the
# mix_hard_negs.py in-context dataset, half our labeled negatives (graded) and half nv-mined
# ones labeled at the easy distance (one-hot). Two groups: ours-nv-mixed (one shuffle over the
# mix) and ours-nv-ordered (train.py --train-order mined-first: each epoch trains every batch of
# the nv-mined queries before any batch of ours). The training distribution changed, so V is
# re-swept on the in-context validation split per group; MIXED_SELECTED (section 3) must equal
# each group's argmax.
MIXED_MINING = "m0.025_s10"
MIXED_REPHRASE = "in-context"
# label -> train.py --train-order token ("" is the default single shuffle)
MIXED_GROUPS = {"ours-nv-mixed": "", "ours-nv-ordered": "mined-first"}
mixed_sweep = {}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, modality in zip(axes, ["text", "multimodal"]):
    for label, order in MIXED_GROUPS.items():
        sub = ablation[(ablation["modality"] == modality) & (ablation["style"] == V3_STYLE)
                       & (ablation["negs"] == "mixed") & (ablation["mining"] == MIXED_MINING)
                       & (ablation["rephrase"] == MIXED_REPHRASE) & (ablation["order"] == order)]
        sweep = sub.set_index("V")["recall@k_mod"].reindex(V3_V_LEVELS)
        mixed_sweep[modality, label] = sweep
        ax.plot(V3_V_LEVELS, sweep.to_numpy(), marker="o", label=label)
        if sweep.notna().any():
            best_v = sweep.idxmax()
            ax.annotate(f"V={int(best_v)}: {sweep[best_v]:.3f}", (best_v, sweep[best_v]),
                        textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
        missing = [int(v) for v in V3_V_LEVELS if pd.isna(sweep.get(v))]
        if missing:
            print(f"{modality} {label}: missing V {missing}")
    ax.set_xscale("log", base=2)
    ax.set_xticks(V3_V_LEVELS, [str(int(v)) for v in V3_V_LEVELS])
    ax.set_xlabel("V")
    ax.set_ylabel(f"Recall@{k_for(modality)} (val)")
    ax.set_title("text" if modality == "text" else "image", fontsize=10)
    ax.legend(fontsize=8)
fig.suptitle(f"{V3_STYLE} on 50/50 mixed negatives \u2014 V sweep ({ABLATION_KIND}{IC_SUFFIX}, validation split)",
             fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "v_sweep_infonce_ours_v3_mixed.png"), dpi=150, bbox_inches="tight")
plt.show()


def mixed_val_argmax(modality, label):
    """V with the best validation recall over one mixed group's sweep; raises on an incomplete sweep."""
    sweep = mixed_sweep[modality, label]
    absent = [int(v) for v in V3_V_LEVELS if pd.isna(sweep.get(v))]
    if absent:
        raise ValueError(f"{modality} {label}: sweep incomplete, no preds_val for V={absent}")
    return int(sweep.idxmax())


In [ ]:
# NV-Retriever mining sweep for the infonce-mined baseline (paper.sh "NV-Retriever mining
# sweep", rephrased, validation split): relative margin x survivor choice. The loss has no
# hparams, so the sweep is over how the negative is mined; every cell is its own
# mine_hard_negs.py --variant dataset, m<margin>_s<skip>, and the default dataset is the
# (0.05, s0) cell. A cell without preds_val is drawn as a hatched placeholder.
NV_MARGINS = [0.025, 0.05, 0.1, 0.2]
NV_SKIPS = [0, 10]
NV_DEFAULT = (0.05, 0)


def nv_cell(variant):
    """(margin, skip) of a mining variant; the empty variant is the default config."""
    if variant == "":
        return NV_DEFAULT
    margin, _, skip = variant.partition("_")
    return float(margin[1:]), int(skip[1:])


nv_runs = val_runs[(val_runs["style"] == "infonce-mined") & (val_runs["negs"] == "mined")
                   & (val_runs["query_kind"] == ABLATION_KIND) & (val_runs["rephrase"] == "")]
nv_values = {}
for run in nv_runs.itertuples():
    row = val_condition(run)
    if row is not None:
        nv_values[run.modality, nv_cell(run.mining)] = row["recall@k_mod"]
print(f"{len(nv_values)} of {2 * len(NV_MARGINS) * len(NV_SKIPS)} mining-sweep cells have preds_val")


def nv_val_argmax(modality):
    """(margin, skip) with the best validation recall; raises on an incomplete grid."""
    cells = {(m, s): nv_values[modality, (m, s)] for m in NV_MARGINS for s in NV_SKIPS
             if (modality, (m, s)) in nv_values}
    absent = [(m, s) for m in NV_MARGINS for s in NV_SKIPS if (m, s) not in cells]
    if absent:
        raise ValueError(f"{modality}: mining sweep incomplete, no preds_val for {absent}")
    return max(cells, key=cells.__getitem__)


def nv_variant(margin, skip):
    """The mine_hard_negs.py --variant tag of a cell; the default cell has none."""
    return "" if (margin, skip) == NV_DEFAULT else f"m{margin}_s{skip}"

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, modality in zip(axes, ["text", "multimodal"]):
    values = np.full((len(NV_SKIPS), len(NV_MARGINS)), np.nan)
    for i, skip in enumerate(NV_SKIPS):
        for j, margin in enumerate(NV_MARGINS):
            if (modality, (margin, skip)) in nv_values:
                values[i, j] = nv_values[modality, (margin, skip)]
    done = ~np.isnan(values)
    lo, hi = (values[done].min(), values[done].max()) if done.any() else (0.0, 1.0)
    ax.imshow(np.where(done, values, lo), cmap="viridis", vmin=lo, vmax=hi, aspect="auto")
    ax.grid(False)
    best = np.nanargmax(values) if done.any() else None
    for i, skip in enumerate(NV_SKIPS):
        for j, margin in enumerate(NV_MARGINS):
            if done[i, j]:
                ax.text(j, i, f"{values[i, j]:.3f}", ha="center", va="center", fontsize=9,
                        fontweight="bold" if i * len(NV_MARGINS) + j == best else "normal",
                        color="white" if values[i, j] < (lo + hi) / 2 else "black")
            else:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="0.9",
                                           edgecolor="0.6", hatch="///", linewidth=0))
                ax.text(j, i, "pending", ha="center", va="center", fontsize=7, color="0.3")
    ax.set_xticks(range(len(NV_MARGINS)), [str(m) for m in NV_MARGINS])
    ax.set_yticks(range(len(NV_SKIPS)), ["first survivor", "skip top 10"])
    ax.set_xlabel("relative margin")
    ax.set_title("text" if modality == "text" else "image", fontsize=10)
fig.suptitle(f"infonce-mined on retrieval-mined negatives — mining sweep "
             f"({ABLATION_KIND}, validation split; Recall@{K_TEXT} text / Recall@{K_IMAGE} image)",
             fontweight="bold", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "nv_mining_sweep.png"), dpi=150, bbox_inches="tight")
plt.show()


## 3. Protocol comparison: graded vs. ungraded, per family

The claim under test is that **grading** the mined hard negative beats leaving it ungraded,
with everything else held fixed -- same batches, same seen examples, same number of
comparisons, targets the only difference. So each family is a designed pair:

| family | ungraded (mined) | graded |
|---|---|---|
| infonce | `infonce-mined` | `infonce-ours-v3` |
| mse | `mse-mined` | `ours-mse-batched` |
| cosent | `cosent` | `ours-cosent` |
| siglip | `siglip-mined` | `ours-siglip` |

Families are in priority order. `infonce-ours-v3` grades the target (mass e^{-s d/V} on
the hard negative); it replaced `ours-infonce-margin` here on 2026-09-03 (margin's easy x V
sweep stays in section 2). `ours-cosent` is a three-rank ordinal (positive > hard >
random) rather than a distance grading: CoSENT reads label order only, so it has no V or
easy and needs no search. The graded member's hparams come from the validation
easy x V sweep (section 2); the comparison itself is on the **test** split.

**Human queries** are the third bar of each group: the model that trained on rephrased
queries (`HUMAN_TRAIN_KIND`), scored on the human-written queries from `preds_human/`, ranked
against that model's own test-split corpus plus the human pairs' products. The human pairs were
sampled before the leakage split, so most of them sit in the base train split; the human bar is
therefore not a clean held-out number until those pairs are moved to test and the models
retrained.

**Which rows are plotted** is read from `paper.sh` itself: its non-`split=val` condition rows
are the main grid, at each style's chosen (V, easy). That keeps this figure in step with what
the sweep actually trains and tests, instead of a hand-maintained slice that goes stale when
hparams change. A slot with no result is **drawn as a hatched placeholder with the reason** --
`hparam selection pending` (the slot's graded style has no completed, current validation
sweep behind it -- see `SELECTED` in the cell below), `not in paper.sh` (no condition row)
or `no preds` (row exists, run not finished) -- rather than silently dropped, so a missing
model is visible in the figure. A graded slot is reported only when `SELECTED` names its
(V, easy), and the cell checks that entry against both the sweep argmax and the paper.sh row.

Groups are ordered by mean Recall (Recall@5 text, Recall@20 image) across **synthetic and
rephrased only**, best first; `original` is excluded from the ordering as a non-primary
metric. Missing slots sort last. Section 4 uses the identical rule.


In [ ]:
from utils.paper_analysis import parse_conditions

# Protocol pairs, in priority order: (ungraded mined loss, its graded counterpart).
PROTOCOL = {
    "infonce": ("infonce-mined", "infonce-ours-v3"),
    "mse":     ("mse-mined",     "ours-mse-batched"),
    "cosent":  ("cosent",        "ours-cosent"),
    "siglip":  ("siglip-mined",  "ours-siglip"),
}
# The retrieval-mined (NV-Retriever) baseline: infonce-mined trained on the mine_hard_negs.py
# negatives instead of ours. One model per slot, drawn beside every family as the reference.
NV_MINED = "nv-mined"
ROLE = {NV_MINED: "baseline"}
# Mining config the nv-mined baseline uses per slot, selected on the sweep in section 2 the
# way SELECTED is for the graded styles: the entry must equal the sweep argmax and the
# paper.sh row must carry the same mining= tag. The sweep covers rephrased only; text
# original stays on the default config, which is not a selection and is marked so.
NV_SELECTED = {
    ("text",       "rephrased"): "m0.025_s10",
    ("multimodal", "rephrased"): "m0.025_s10",
    ("text",       "original"):  None,   # not swept; the default dataset, drawn as pending
}
for family, (ungraded, graded) in PROTOCOL.items():
    ROLE[ungraded] = "ungraded"
    ROLE[graded] = "graded"
# The 50/50 labeled/nv-mined mixes (mix_hard_negs.py, in-context only): infonce-ours-v3 with
# our graded negative on half the hard rows and the nv-mined one, at the easy label, on the
# other half. Drawn in the infonce panel beside nv-mined; labels match MIXED_GROUPS in
# section 2. V per group is selected on its in-context validation sweep and gated like
# SELECTED: the entry must equal the sweep argmax and the paper.sh 3-seed rows must carry it.
# 2026-09-12: sweep complete; argmax at recall@5 text / recall@20 image, both groups.
MIXED_SELECTED = {
    ("text",       "rephrased", "ours-nv-mixed"):   20,
    ("multimodal", "rephrased", "ours-nv-mixed"):   80,
    ("text",       "rephrased", "ours-nv-ordered"): 20,
    ("multimodal", "rephrased", "ours-nv-ordered"): 80,
}
for label in MIXED_GROUPS:
    ROLE[label] = "mixed"
# 2026-09-09: synthetic rows are commented out in paper.sh and not shown; original (text
# only) and rephrased remain. Ranking is on rephrased, the kind both modalities have.
# human is scored on the model trained on HUMAN_TRAIN_KIND queries (the paper's headline
# kind, which both modalities have); results rows carry train_kind to tell that apart.
QK_ORDER = ["original", "rephrased", "rephrased" + IC_SUFFIX, "human", "human" + IC_SUFFIX]
RANK_KINDS = ["rephrased"]
HUMAN_TRAIN_KIND = "rephrased"


def train_kind_of(kind):
    """The query kind a slot's model trained on: the kind itself, or HUMAN_TRAIN_KIND for human;
    the -ic kinds are the rephrased-trained model on the in-context dataset."""
    kind = kind.removesuffix(IC_SUFFIX)
    return HUMAN_TRAIN_KIND if kind == "human" else kind


def rephrase_of(kind):
    """The rephrase= variant of the paper.sh row a scored kind comes from."""
    return "in-context" if kind.endswith(IC_SUFFIX) else ""

qk_colour = dict(zip(QK_ORDER, sns.color_palette("colorblind", len(QK_ORDER))))


def easy_of(extra):
    for token in (extra or "").split(","):
        if token.startswith("easy="):
            return int(token[len("easy="):])
    return 20


# The main grid is whatever paper.sh will train and test: its non-val rows, at each style's
# chosen hparams. Read from the script so this figure cannot drift from the sweep.
main_grid = parse_conditions("paper.sh")
main_grid = main_grid[~main_grid["extra"].fillna("").str.contains("split=val")].copy()
# The protocol compares grading against not grading the *labeled* negative; rows trained on
# retrieval-mined negatives (negs=mined) are a separate comparison and are excluded here.
main_grid = main_grid[main_grid["negs"] == "labeled"].copy()
main_grid["easy"] = main_grid["extra"].map(easy_of)


# Hyperparameters selected on the validation sweep (section 2), one entry per graded slot
# whose main-grid row depends on that sweep. paper.sh trains whatever V/easy its row says;
# this table says which of those rows are backed by a completed, current sweep. A slot whose
# entry is None is drawn as the hatched placeholder, not as a result, whatever paper.sh holds.
# When making changes that invalidate model selection (a loss, a dataset, the selection
# metric, or the sweep protocol), reset the affected entries to `None` until a proper hparam
# sweep has completed. Only then select the model: set the entry to the sweep argmax and put
# the same V/easy on the paper.sh row.
#
# 2026-09-05: per-query-kind sweep complete (paper.sh "Per-query-kind hparam search"); every
# slot is the argmax of its own validation grid at recall@5 text / recall@20 image, and
# paper.sh carries the same V/easy on each main-grid row.
SELECTED = {
    ("text",       "infonce-ours-v3",  "original"):  {"V": 10, "easy": 20},
    # ("text",       "infonce-ours-v3",  "synthetic"): {"V": 20, "easy": 20},
    ("text",       "infonce-ours-v3",  "rephrased"): {"V": 20, "easy": 20},
    # ("multimodal", "infonce-ours-v3",  "synthetic"): {"V": 10, "easy": 20},
    ("multimodal", "infonce-ours-v3",  "rephrased"): {"V": 10, "easy": 20},
    ("text",       "ours-mse-batched", "original"):  {"V": 40, "easy": 10},
    # ("text",       "ours-mse-batched", "synthetic"): {"V": 40, "easy": 10},
    ("text",       "ours-mse-batched", "rephrased"): {"V": 40, "easy": 10},
    # ("multimodal", "ours-mse-batched", "synthetic"): {"V": 80, "easy": 10},
    ("multimodal", "ours-mse-batched", "rephrased"): {"V": 20, "easy": 10},
    ("text",       "ours-siglip",      "original"):  {"V": 20, "easy": 10},
    # ("text",       "ours-siglip",      "synthetic"): {"V": 20, "easy": 10},
    ("text",       "ours-siglip",      "rephrased"): {"V": 20, "easy": 10},
    # ("multimodal", "ours-siglip",      "synthetic"): {"V": 20, "easy": 10},
    ("multimodal", "ours-siglip",      "rephrased"): {"V": 40, "easy": 10},
}
# The sweep grid each style is selected over. infonce-ours-v3 has no easy axis, so its
# grid is V alone at the default easy (see section 2).
SWEEP_GRID = {
    "ours-mse-batched": [(easy, V) for easy in EASY_LEVELS for V in V_LEVELS],
    "ours-siglip":      [(easy, V) for easy in EASY_LEVELS for V in V_LEVELS],
    V3_STYLE:           [(20, V) for V in V3_V_LEVELS],
}


def val_argmax(modality, style, kind):
    """(easy, V) with the best validation recall over the style's sweep grid for this query
    kind; raises if any grid cell has no preds_val, since a selection needs the whole grid."""
    cells = {}
    for run in val_runs[(val_runs["modality"] == modality) & (val_runs["style"] == style)
                        & (val_runs["query_kind"] == kind) & (val_runs["negs"] == "labeled")
                        & (val_runs["rephrase"] == "")].itertuples():
        row = val_condition(run)
        if row is not None:
            cells[int(run.easy), float(run.V)] = row["recall@k_mod"]
    grid = SWEEP_GRID[style]
    absent = [cell for cell in grid if cell not in cells]
    if absent:
        raise ValueError(f"{modality} {style} {kind}: sweep incomplete, no preds_val for {absent}")
    return max(grid, key=lambda cell: cells[cell])


def main_row(modality, style, kind):
    """The test-split results row for one protocol slot, or (None, why it is missing)."""
    train_kind = train_kind_of(kind)
    if style in SWEEP_GRID:
        chosen = SELECTED[modality, style, train_kind]
        if chosen is None:
            return None, "hparam selection pending"
        best_easy, best_v = val_argmax(modality, style, train_kind)
        if (chosen["easy"], float(chosen["V"])) != (best_easy, best_v):
            raise ValueError(f"{modality} {style} {train_kind}: SELECTED says {chosen}, "
                             f"sweep argmax is easy={best_easy}, V={int(best_v)}")
    cond = main_grid[(main_grid["modality"] == modality) & (main_grid["style"] == style)
                     & (main_grid["query_kind"] == train_kind) & (main_grid["rephrase"] == rephrase_of(kind))]
    if cond.empty:
        return None, "not in paper.sh"
    c = cond.iloc[0]
    if style in SWEEP_GRID and (c["easy"], float(c["V"])) != (chosen["easy"], float(chosen["V"])):
        raise ValueError(f"{modality} {style} {train_kind}: paper.sh row has easy={c['easy']}, "
                         f"V={int(c['V'])}, SELECTED says {chosen}")
    same_v = results["V"].isna() if pd.isna(c["V"]) else (results["V"] == c["V"])
    hit = results[(results["modality"] == modality) & (results["style"] == style)
                  & (results["query_kind"] == kind) & (results["train_kind"] == train_kind)
                  & (results["easy"] == c["easy"]) & same_v
                  & (results["negs"] == "labeled") & (results["rephrase"] == rephrase_of(kind))]
    return trials_or_why(hit, cond)


def trials_or_why(hit, cond):
    """(trials, note) for a condition: its healthy trial rows, or (None, why) when there are
    none. When fewer seeds than paper.sh declares are done, the rows are returned and the
    note says so; the bar is drawn from the seeds that exist and carries the note."""
    declared = int(cond["seed"].nunique())
    if hit.empty:
        return None, "no preds"
    if len(hit) < declared:
        return hit, f"{len(hit)}/{declared} trials"
    return hit, None


def trial_stats(frame, k):
    """mean over a condition's trials, and the number of trials."""
    values = frame[f"recall@{k}"].to_numpy(dtype=float)
    return float(values.mean()), len(values)


def kinds_for(modality):
    """Query kinds the main grid defines for this modality (image has no `original`), plus
    human whenever the model it is scored on (HUMAN_TRAIN_KIND) is in the grid."""
    grid = main_grid[main_grid["modality"] == modality]
    present = set(grid[grid["rephrase"] == ""]["query_kind"])
    if HUMAN_TRAIN_KIND in present:
        present.add("human")
    # in-context rows exist for this modality: the same model kinds, scored on the in-context data
    if (grid["rephrase"] == "in-context").any():
        present.update({"rephrased" + IC_SUFFIX, "human" + IC_SUFFIX})
    return [k for k in QK_ORDER if k in present]


rows, missing, incomplete = [], {}, {}
for modality in ["text", "multimodal"]:
    for ungraded, graded in PROTOCOL.values():
        for style in (ungraded, graded):
            for kind in kinds_for(modality):
                trials, note = main_row(modality, style, kind)
                if trials is None:
                    missing[modality, style, kind] = note
                    continue
                rows.append(trials)
                if note:
                    incomplete[modality, style, kind] = note
main = pd.concat(rows).reset_index(drop=True)
n_slots = len(main.drop_duplicates(["modality", "style", "query_kind"]))
print(f"{n_slots} protocol slots with results ({len(main)} trials), {len(missing)} missing, "
      f"{len(incomplete)} with fewer seeds than declared")


def rank_kinds_of(modality, style, value_of):
    """The kinds a group is ranked on: RANK_KINDS, or their -ic counterparts for a group that
    exists only in-context (the mixed groups)."""
    kinds = kinds_for(modality)
    plain = [k for k in RANK_KINDS if k in kinds]
    if all(value_of(modality, style, k) is not None for k in plain):
        return plain
    return [k + IC_SUFFIX for k in RANK_KINDS if k + IC_SUFFIX in kinds]


def ordered_groups(modality, styles, value_of):
    """Groups sorted by mean value over their rank kinds, best first; missing last."""
    kinds = kinds_for(modality)

    def key(style):
        vals = [value_of(modality, style, k) for k in rank_kinds_of(modality, style, value_of)]
        return -sum(vals) / len(vals) if vals and all(v is not None for v in vals) else float("inf")
    return sorted(styles, key=key), kinds


def cell_x(i, j, width, kinds):
    """x of the bar for group i, kind j."""
    return i + (j - (len(kinds) - 1) / 2) * width


def label_cell(ax, x, text):
    """Vertical note on one bar cell (a missing reason, or a fewer-seeds-than-declared note)."""
    ax.text(x, 0.5, text, rotation=90, ha="center", va="center", fontsize=7,
            color="0.3", transform=ax.get_xaxis_transform(), zorder=3)


def draw_missing(ax, i, j, width, kinds, reason):
    """Hatched placeholder for one empty (group, kind) cell, labeled with its reason."""
    x = cell_x(i, j, width, kinds)
    ax.axvspan(x - width * 0.475, x + width * 0.475, facecolor="0.9", edgecolor="0.6",
               hatch="///", linewidth=0, zorder=0)
    label_cell(ax, x, reason)


grid_all_mined = parse_conditions("paper.sh")
grid_all_mined = grid_all_mined[~grid_all_mined["extra"].fillna("").str.contains("split=val")
                                & (grid_all_mined["style"] == "infonce-mined")
                                & (grid_all_mined["negs"] == "mined")]


def nv_row(modality, kind):
    """The nv-mined baseline's test row for one slot, or (None, why it is missing)."""
    train_kind = train_kind_of(kind)
    chosen = NV_SELECTED[modality, train_kind]
    if chosen is None:
        return None, "mining not swept"
    best = nv_variant(*nv_val_argmax(modality))
    if chosen != best:
        raise ValueError(f"{modality}: NV_SELECTED says {chosen!r}, sweep argmax is {best!r}")
    cond = grid_all_mined[(grid_all_mined["modality"] == modality) & (grid_all_mined["query_kind"] == train_kind)
                          & (grid_all_mined["rephrase"] == rephrase_of(kind))]
    if cond.empty or cond["mining"].iloc[0] != chosen:
        return None, f"paper.sh row is not mining={chosen}"
    hit = results[(results["modality"] == modality) & (results["style"] == "infonce-mined")
                  & (results["query_kind"] == kind) & (results["train_kind"] == train_kind)
                  & (results["negs"] == "mined") & (results["mining"] == chosen)
                  & (results["rephrase"] == rephrase_of(kind))]
    return trials_or_why(hit, cond)


grid_mixed = parse_conditions("paper.sh")
grid_mixed = grid_mixed[~grid_mixed["extra"].fillna("").str.contains("split=val")
                        & (grid_mixed["style"] == V3_STYLE) & (grid_mixed["negs"] == "mixed")].copy()
grid_mixed["easy"] = grid_mixed["extra"].map(easy_of)


def mixed_row(modality, label, kind):
    """One mixed group's test row for a slot, or (None, why it is missing)."""
    if rephrase_of(kind) != MIXED_REPHRASE:
        return None, "in-context only"
    train_kind = train_kind_of(kind)
    order = MIXED_GROUPS[label]
    chosen = MIXED_SELECTED[modality, train_kind, label]
    if chosen is None:
        return None, "hparam selection pending"
    best_v = mixed_val_argmax(modality, label)
    if chosen != best_v:
        raise ValueError(f"{modality} {label}: MIXED_SELECTED says V={chosen}, sweep argmax is V={best_v}")
    cond = grid_mixed[(grid_mixed["modality"] == modality) & (grid_mixed["query_kind"] == train_kind)
                      & (grid_mixed["order"] == order) & (grid_mixed["rephrase"] == MIXED_REPHRASE)]
    if cond.empty:
        return None, "not in paper.sh"
    c = cond.iloc[0]
    if int(c["V"]) != chosen:
        raise ValueError(f"{modality} {label}: paper.sh row has V={int(c['V'])}, MIXED_SELECTED says V={chosen}")
    hit = results[(results["modality"] == modality) & (results["style"] == V3_STYLE)
                  & (results["query_kind"] == kind) & (results["train_kind"] == train_kind)
                  & (results["easy"] == c["easy"]) & (results["V"] == c["V"])
                  & (results["negs"] == "mixed") & (results["mining"] == c["mining"])
                  & (results["order"] == order) & (results["rephrase"] == MIXED_REPHRASE)]
    return trials_or_why(hit, cond)


def stats_of(modality, style, kind):
    """(mean, n) over a slot's trials, or None if the slot is missing."""
    if style == NV_MINED:
        trials, _ = nv_row(modality, kind)
    elif style in MIXED_GROUPS:
        trials, _ = mixed_row(modality, style, kind)
    else:
        trials = main[(main["modality"] == modality) & (main["style"] == style) & (main["query_kind"] == kind)]
        trials = None if trials.empty else trials
    return None if trials is None else trial_stats(trials, k_for(modality))


def recall_of(modality, style, kind):
    st = stats_of(modality, style, kind)
    return None if st is None else st[0]


for modality in ["text", "multimodal"]:
    for kind in kinds_for(modality):
        for group, row_of in [(NV_MINED, nv_row)] + [(g, lambda m, k, g=g: mixed_row(m, g, k)) for g in MIXED_GROUPS]:
            trials, note = row_of(modality, kind)
            if trials is None:
                missing[modality, group, kind] = note
            elif note:
                incomplete[modality, group, kind] = note


def print_ranking(modality, groups, value_of, stats_for=None):
    """Groups best first by mean recall over RANK_KINDS, as three tables with the same rows
    and columns: the mean of every kind of the modality ('-' when that kind is missing),
    then the number of trials. A group missing a
    RANK_KINDS value is listed as missing in the first table and omitted from the others."""
    # original is non-primary (excluded from the ranking), so it prints last
    show_kinds = sorted(kinds_for(modality), key=lambda k: k == "original")
    w = 13

    def table(title, cell):
        print(f"  {modality}: {title}")
        print(f"    {'group':<26}" + "".join(f"{kind:>{w}}" for kind in show_kinds))
        for g in groups:
            order_kinds = rank_kinds_of(modality, g, value_of)
            if not order_kinds or any(value_of(modality, g, kind) is None for kind in order_kinds):
                if title.startswith("Recall"):
                    print(f"    {g:<26}missing")
                continue
            print(f"    {g:<26}" + "".join(f"{cell(g, kind):>{w}}" for kind in show_kinds))

    def fmt(g, kind, pick):
        if value_of(modality, g, kind) is None:
            return "-"
        return pick(*stats_for(modality, g, kind))

    table(f"Recall@{k_for(modality)}", lambda g, kind: fmt(g, kind, lambda mean, n: f"{mean:.4f}"))
    if stats_for:
        table("n trials", lambda g, kind: fmt(g, kind, lambda mean, n: str(n)))


for family, pair in PROTOCOL.items():
    print(f"{family}:")
    extra = [NV_MINED] + (list(MIXED_GROUPS) if family == "infonce" else [])
    fig, axes = plt.subplots(1, 2, figsize=(max(11, 3.2 * (len(pair) + len(extra))), 4.2))
    for ax, modality in zip(axes, ["text", "multimodal"]):
        groups, kinds = ordered_groups(modality, list(pair) + extra, recall_of)
        print_ranking(modality, groups, recall_of, stats_of)
        width = 0.8 / len(kinds)
        for j, k in enumerate(kinds):
            xs = [cell_x(i, j, width, kinds) for i in range(len(groups))]
            sts = [stats_of(modality, s, k) for s in groups]
            ys = [st[0] if st is not None else None for st in sts]
            ax.bar(xs, [y if y is not None else 0 for y in ys], width=width * 0.95,
                   color=qk_colour[k], label=k)
            for i, (s, y) in enumerate(zip(groups, ys)):
                if y is None:
                    draw_missing(ax, i, j, width, kinds, missing[modality, s, k])
                elif (modality, s, k) in incomplete:
                    label_cell(ax, xs[i], incomplete[modality, s, k])
        if not any(recall_of(modality, s, k) is not None for s in groups for k in kinds):
            ax.set_ylim(0, 1)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([f"{s}\n({ROLE[s]}, n={max(st[1] for st in (stats_of(modality, s, k) for k in kinds) if st)})"
                            if any(stats_of(modality, s, k) for k in kinds) else f"{s}\n({ROLE[s]})"
                            for s in groups], fontsize=8)
        ax.set_title("text" if modality == "text" else "image")
        ax.set_ylabel(f"Recall@{k_for(modality)}")
    axes[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=5, frameon=False)
    fig.suptitle(f"{family}: ungraded vs graded vs nv-mined baseline (test split)", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"protocol_{family}.png"), dpi=150)
    plt.show()

if missing:
    print("missing protocol slots:")
    for (modality, style, kind), why in sorted(missing.items()):
        print(f"  {modality:10} {style:20} {kind:10} {why}")


## 4. Baseline loss function selection

Which ungraded loss is strongest under baseline conditions, to justify building on InfoNCE.
Rows are the ungraded losses, trained on our labeled hard negative without its distance;
columns are the in-context slot of each modality, scored on the **validation** split
(paper.sh "Baseline loss selection" rows, `preds_val/`), so the baseline is not chosen on
the test set. Each cell is the mean over trials. `mse` is pairwise MSE of cos(q, x) onto 1/0
(CosineSimilarityLoss, the ungraded control of `classic-mse`; paper.sh rows added
2026-09-12). None of the four losses has a hyperparameter, so there is nothing to sweep; the
graded counterparts' sweeps are in section 2. `infonce+nv` is `infonce-mined` trained on the
NV-Retriever-mined in-context negatives at the mining variant `NV_SELECTED` picked in section 3,
so the table also shows whether our labeled negatives or retrieval-mined ones make the
stronger InfoNCE baseline.

The cell also writes `paper/figs/baseline_loss_table.tex`, the tabular that main.tex
`\input`s as Table `tab:baseline-loss`, so the paper compiles from the current results
without hand-typed numbers. Rows are ordered by text-ic ascending, the best value per column
is bold, a slot with fewer than `BASELINE_SEEDS` = 3 trials prints in red with its `(n=…)`,
and a slot with no result prints a red `pending`, so an incomplete table is visible in the
PDF rather than breaking the build or passing as final.


In [ ]:
# label -> (style, negs, mining). negs/mining name the training dataset: labeled = our
# synthesized hard negative; mined = the NV-Retriever-mined one at the selected mining variant.
BASELINE_LOSSES = {"mse": ("mse", "labeled", ""), "cosent": ("cosent", "labeled", ""),
                   "siglip": ("siglip-mined", "labeled", ""), "infonce": ("infonce-mined", "labeled", ""),
                   "infonce+nv": ("infonce-mined", "mined", NV_SELECTED["text", "rephrased"])}
BASELINE_COLUMNS = {"text-ic": "text", "image-ic": "multimodal"}
BASELINE_TRAIN_KIND = "rephrased"
BASELINE_REPHRASE = "in-context"


def baseline_val_stats(modality, style, negs, mining):
    """(mean validation recall, n trials) over the seeds of one baseline slot, or (None, why).
    Reads preds_val/ of the in-context run dirs (val_runs, section 2)."""
    runs = val_runs[(val_runs["modality"] == modality) & (val_runs["style"] == style)
                    & (val_runs["query_kind"] == BASELINE_TRAIN_KIND) & (val_runs["rephrase"] == BASELINE_REPHRASE)
                    & (val_runs["negs"] == negs) & (val_runs["mining"] == mining) & (val_runs["order"] == "")]
    if runs.empty:
        return None, "no run dir"
    values = [row["recall@k_mod"] for row in (val_condition(r) for r in runs.itertuples()) if row is not None]
    if not values:
        return None, "no preds_val"
    return (float(np.mean(values)), len(values)), None


rows, baseline_missing = [], {}
for loss, (style, negs, mining) in BASELINE_LOSSES.items():
    row = {"loss": loss}
    for column, modality in BASELINE_COLUMNS.items():
        st, note = baseline_val_stats(modality, style, negs, mining)
        if st is None:
            baseline_missing[loss, column] = note
            row[f"{column} Recall@{k_for(modality)}"] = np.nan
            row[f"{column} n"] = 0
            continue
        row[f"{column} Recall@{k_for(modality)}"], row[f"{column} n"] = st
    rows.append(row)
baseline_table = pd.DataFrame(rows).set_index("loss")
display(baseline_table.round(4))
for column, modality in BASELINE_COLUMNS.items():
    print(f"{column}: best = {baseline_table[f'{column} Recall@{k_for(modality)}'].idxmax()}")
for (loss, column), note in baseline_missing.items():
    print(f"missing: {loss} {column}: {note}")

# LaTeX table for the paper. Only the tabular lives here; the table environment, caption and
# label are in main.tex so the prose is edited there.
BASELINE_TEX_NAMES = {"mse": "MSE", "cosent": "CoSENT", "siglip": "SigLIP", "infonce": "InfoNCE",
                      "infonce+nv": "InfoNCE + NV"}
recall_columns = [f"{column} Recall@{k_for(modality)}" for column, modality in BASELINE_COLUMNS.items()]
ordered = baseline_table.sort_values(recall_columns[0], ascending=True, na_position="first")
best = {column: ordered[column].idxmax() for column in recall_columns}


BASELINE_SEEDS = 3   # trials paper.sh declares per slot; fewer prints red with the count


def tex_cell(loss, column):
    value = ordered.loc[loss, column]
    if np.isnan(value):
        return r"\textcolor{red}{pending}"
    text = f"{value:.3f}"
    if loss == best[column]:
        text = r"\textbf{" + text + "}"
    n = int(ordered.loc[loss, column.split(" Recall@")[0] + " n"])
    if n < BASELINE_SEEDS:
        text = r"\textcolor{red}{" + text + f" (n={n})" + "}"
    return text


lines = [r"\begin{tabular}{lrr}", r"    \toprule",
         "    Loss & Text (Recall@%d) & Image (Recall@%d) \\\\" % (K_TEXT, K_IMAGE), r"    \midrule"]
for loss in ordered.index:
    lines.append("    " + " & ".join([BASELINE_TEX_NAMES[loss]] + [tex_cell(loss, c) for c in recall_columns]) + r" \\")
lines += [r"    \bottomrule", r"\end{tabular}"]
tex_path = os.path.join(FIG_DIR, "baseline_loss_table.tex")
with open(tex_path, "w") as f:
    f.write("\n".join(lines) + "\n")
print(f"wrote {tex_path}")
print(open(tex_path).read())


## 5. Win rate: positive vs. hard negative

Pairwise accuracy: for each (query, positive, hard negative) triple the model **wins** when it
scores the positive above the negative. `sim_pos`/`sim_neg` are already stored per row in every
run's `preds/triplets.jsonl`, so this is a read of existing preds, not a re-encode.

Easy negatives (`negative_example_source == "random"`) are excluded -- they are the random
distractors, not the mined hard negative this experiment is about. Ties count as losses; the
tie count is printed below so it stays visible rather than assumed negligible.

Same protocol pairs, same main-grid rows (read from `paper.sh`), same missing-slot
placeholders, the same `human` bar (rephrased-trained model, `preds_human/triplets.jsonl`)
and the **same group ordering as section 3**: mean Recall across synthetic and
rephrased, `original` excluded as non-primary. So panels line up bar-for-bar with section 3
and the two metrics can be read across. The y-axis starts at chance (0.5), so bar height is
the margin above a coin flip.


In [ ]:
from utils.distance_labels import EASY_NEGATIVE_SOURCE


def win_rate(run_dir, preds_subdir="preds"):
    """Fraction of hard-negative pairs whose positive scores above the negative."""
    with open(os.path.join(run_dir, preds_subdir, "triplets.jsonl"), encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    hard = [r for r in rows if r["negative_example_source"] != EASY_NEGATIVE_SOURCE]
    return {"win_rate": sum(r["sim_pos"] > r["sim_neg"] for r in hard) / len(hard),
            "n_pairs": len(hard),
            "n_ties": sum(r["sim_pos"] == r["sim_neg"] for r in hard)}


wins = pd.DataFrame([
    {"modality": r.modality, "style": r.style, "query_kind": scored_kind(r, "preds"), "train_kind": r.query_kind,
     "rephrase": r.rephrase, "V": r.V, "easy": r.easy, **win_rate(r.run_dir)}
    for r in usable.itertuples()
] + [
    {"modality": r.modality, "style": r.style, "query_kind": scored_kind(r, "preds_human"), "train_kind": r.query_kind,
     "rephrase": r.rephrase, "V": r.V, "easy": r.easy, **win_rate(r.run_dir, "preds_human")}
    for r in usable_human.itertuples()
])
print(f"{len(wins)} conditions | {wins['n_pairs'].sum():,} hard-negative pairs | "
      f"{wins['n_ties'].sum():,} exact ties (counted as losses)")


In [ ]:
# `main`, `PROTOCOL`, `missing` and the ordering helpers are reused from section 3 so the
# grouping and bar order here cannot drift from that figure.
wins_main = main[["modality", "style", "query_kind", "train_kind", "rephrase", "V", "easy"]].merge(
    wins, on=["modality", "style", "query_kind", "train_kind", "rephrase", "V", "easy"], how="left")
wr = {(r.modality, r.style, r.query_kind): r.win_rate
      for r in wins_main.itertuples() if not pd.isna(r.win_rate)}


def win_of(modality, style, kind):
    return wr[modality, style, kind] if (modality, style, kind) in wr else None


for family, pair in PROTOCOL.items():
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, modality in zip(axes, ["text", "multimodal"]):
        # Ordered by recall, exactly as section 3, so the panels align.
        groups, kinds = ordered_groups(modality, list(pair), recall_of)
        width = 0.8 / len(kinds)
        for j, k in enumerate(kinds):
            xs = [cell_x(i, j, width, kinds) for i in range(len(groups))]
            ys = [win_of(modality, s, k) for s in groups]
            ax.bar(xs, [y if y is not None else 0 for y in ys], width=width * 0.95,
                   color=qk_colour[k], label=k)
            for i, (s, y) in enumerate(zip(groups, ys)):
                if y is None:
                    why = missing[modality, s, k] if (modality, s, k) in missing else "no triplets"
                    draw_missing(ax, i, j, width, kinds, why)
                elif (modality, s, k) in incomplete:
                    label_cell(ax, xs[i], incomplete[modality, s, k])
        # Floor at chance rather than 0: a win rate below 0.5 would be worse than a coin
        # flip, so bar height reads as the margin above chance.
        ax.set_ylim(bottom=0.5)
        if not any(win_of(modality, s, k) is not None for s in groups for k in kinds):
            ax.set_ylim(0.5, 1)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([f"{s}\n({ROLE[s]})" for s in groups], fontsize=8)
        ax.set_title("text" if modality == "text" else "image")
        ax.set_ylabel("win rate vs hard negative")
    axes[0].legend(fontsize=8)
    fig.suptitle(f"{family}: ungraded vs graded -- win rate", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"winrate_{family}.png"), dpi=150)
    plt.show()

display(wins_main.pivot_table(index=["modality", "style"], columns="query_kind",
                              values="win_rate", observed=True).round(4))


## 6. Best mining baseline vs. ours

Reviewers will read "mined" as retrieval-mined. This section puts the standard mining
pipeline next to our negatives under one loss, then adds our grading on top:

| group | negatives | loss | what it is |
|---|---|---|---|
| `nv-mined` | retrieval-mined, NV-Retriever rule | `infonce-mined` | the best published miner (Moreira et al., CIKM 2025: strong frozen teacher, top-1000 candidates, drop anything above 95% of the positive's score, keep the best survivor) under the strongest ungraded loss |
| `ours-mined` | our labeled hard negative | `infonce-mined` | same loss, our negative source, ungraded |
| `ours-v3` | our labeled hard negative | `infonce-ours-v3` | same negative source, graded target |

`infonce-mined` is the strongest ungraded loss in section 3, so `nv-mined` is the
best-miner-plus-strongest-standard-loss control in one row per query kind. The three
datasets share validation and test rows exactly (mine_hard_negs.py rewrites the train split
only), so every bar is scored on the same corpus and queries.

Reading the gaps: `ours-mined` − `nv-mined` is the value of our negative *source*;
`ours-v3` − `ours-mined` is the value of *grading*; `ours-v3` − `nv-mined` is the paper's
headline against the standard pipeline. The graded group goes through the same `SELECTED`
gate as section 3. Slots are drawn as hatched placeholders until their rows are enabled in
paper.sh (`negs=mined`, currently commented out pending the mined datasets) and have preds.

The table below the figure reads each mined dataset's `mining_report.json`: how many train
queries got an admissible negative, how many the rule dropped, and how often the mined
product coincides with our labeled negative. The drop rate is the method's own behavior on
attribute-conjunction queries and is reported, not tuned.


In [ ]:
# (label, style, negs, order). negs: labeled = our synthesized negative; mined = the
# retrieval-mined (NV-Retriever) negative, distance unmeasured; mined-graded = the same mined
# negative after label_mined_negs.py measured its distance, so a graded loss can train on it;
# mixed = mix_hard_negs.py, half ours and half nv-mined at the easy label (one-hot).
# order: train.py --train-order ("" = one shuffle; mined-first = every batch of the nv-mined
# queries before any batch of ours, each epoch).
MINING_GROUPS = [
    ("ours graded",   "infonce-ours-v3", "labeled", ""),
    ("ours ungraded", "infonce-mined",   "labeled", ""),
    ("nv-mined",      "infonce-mined",   "mined",   ""),
    # our grading applied to the retrieval-mined negatives (label_mined_negs.py, 2026-09-09)
    ("ours graded + nv-mined",   "infonce-ours-v3",     "mined-graded", ""),
    ("margin graded + nv-mined", "ours-infonce-margin", "mined-graded", ""),
    # the 50/50 mix, in-context only (2026-09-11); labels match MIXED_GROUPS in section 2
    ("ours-nv-mixed",   "infonce-ours-v3", "mixed", ""),
    ("ours-nv-ordered", "infonce-ours-v3", "mixed", "mined-first"),
]
# The mixed groups' V gate (MIXED_SELECTED) and lookup (mixed_row) live in section 3.
# The x-axis orders groups by mean recall over RANK_KINDS (section 3), so the bars read as
# a ranking.

# Every non-val row of paper.sh, mined ones included (section 3's main_grid keeps labeled only).
grid_all = parse_conditions("paper.sh")
grid_all = grid_all[~grid_all["extra"].fillna("").str.contains("split=val")].copy()
grid_all["easy"] = grid_all["extra"].map(easy_of)


def mining_row(modality, style, negs, order, label, kind):
    """Results row for one group, or (None, why). Labeled slots reuse section 3's lookup and
    its hparam-selection gate; mined and mined-graded slots carry the labeled row's hparams
    on their paper.sh line and match on negs directly."""
    if negs == "labeled":
        return main_row(modality, style, kind)
    if negs == "mined":
        return nv_row(modality, kind)
    train_kind = train_kind_of(kind)
    if negs == "mixed":
        return mixed_row(modality, label, kind)
    cond = grid_all[(grid_all["modality"] == modality) & (grid_all["style"] == style)
                    & (grid_all["query_kind"] == train_kind) & (grid_all["negs"] == negs)
                    & (grid_all["order"] == order) & (grid_all["rephrase"] == rephrase_of(kind))]
    if cond.empty:
        return None, "not in paper.sh"
    c = cond.iloc[0]
    same_v = results["V"].isna() if pd.isna(c["V"]) else (results["V"] == c["V"])
    hit = results[(results["modality"] == modality) & (results["style"] == style)
                  & (results["query_kind"] == kind) & (results["train_kind"] == train_kind)
                  & (results["easy"] == c["easy"]) & same_v
                  & (results["negs"] == negs) & (results["mining"] == c["mining"])
                  & (results["order"] == order) & (results["rephrase"] == rephrase_of(kind))]
    return trials_or_why(hit, cond)


mining_values, mining_stats, mining_missing, mining_incomplete = {}, {}, {}, {}
for modality in ["text", "multimodal"]:
    for label, style, negs, order in MINING_GROUPS:
        for kind in kinds_for(modality):
            trials, note = mining_row(modality, style, negs, order, label, kind)
            if trials is None:
                mining_missing[modality, label, kind] = note
                continue
            mining_stats[modality, label, kind] = trial_stats(trials, k_for(modality))
            mining_values[modality, label, kind] = mining_stats[modality, label, kind][0]
            if note:
                mining_incomplete[modality, label, kind] = note
print(f"{len(mining_values)} slots with results, {len(mining_missing)} missing, "
      f"{len(mining_incomplete)} with fewer seeds than declared")

def rank_score(modality, label):
    """Mean recall over RANK_KINDS; a group missing any kind sorts last."""
    vals = [mining_value(modality, label, k) for k in RANK_KINDS]
    return sum(vals) / len(vals) if all(v is not None for v in vals) else float("-inf")


def mining_value(modality, label, kind):
    return mining_values[modality, label, kind] if (modality, label, kind) in mining_values else None


def mining_stat(modality, label, kind):
    return mining_stats[modality, label, kind] if (modality, label, kind) in mining_stats else None


fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, modality in zip(axes, ["text", "multimodal"]):
    groups = sorted([label for label, _, _, _ in MINING_GROUPS],
                    key=lambda g: rank_score(modality, g), reverse=True)
    kinds = kinds_for(modality)
    width = 0.8 / len(kinds)
    for j, k in enumerate(kinds):
        xs = [cell_x(i, j, width, kinds) for i in range(len(groups))]
        ys = [mining_value(modality, g, k) for g in groups]
        ax.bar(xs, [y if y is not None else 0 for y in ys], width=width * 0.95,
               color=qk_colour[k], label=k,
)
        for i, (g, y) in enumerate(zip(groups, ys)):
            if y is None:
                draw_missing(ax, i, j, width, kinds, mining_missing[modality, g, k])
            elif (modality, g, k) in mining_incomplete:
                label_cell(ax, xs[i], mining_incomplete[modality, g, k])
    if not any((modality, g, k) in mining_values for g in groups for k in kinds):
        ax.set_ylim(0, 1)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels([f"{g} (n={max(mining_stats[modality, g, k][1] for k in kinds if (modality, g, k) in mining_stats)})"
                        if any((modality, g, k) in mining_stats for k in kinds) else g for g in groups],
                       fontsize=8, rotation=15)
    ax.set_title("text" if modality == "text" else "image")
    ax.set_ylabel(f"Recall@{k_for(modality)}")
axes[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)
fig.suptitle("Best mining baseline vs. ours (test split)", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "mining_baseline.png"), dpi=150)
plt.show()

if mining_missing:
    print("missing slots:")
    for (modality, label, kind), why in sorted(mining_missing.items()):
        print(f"  {modality:10} {label:11} {kind:10} {why}")

# Ranking, same layout as section 3.
for modality in ["text", "multimodal"]:
    groups = sorted([label for label, _, _, _ in MINING_GROUPS],
                    key=lambda g: rank_score(modality, g), reverse=True)
    print_ranking(modality, groups, mining_value, mining_stat)

# What the miner did to the train split, per mined dataset.
import glob
reports = sorted(glob.glob("dataset/processed/*_mined-*/mining_report.json"))
if reports:
    rows = []
    for path in reports:
        r = json.load(open(path))
        rows.append({
            "dataset": os.path.basename(os.path.dirname(path)),
            "query_kind": r["query_kind"],
            "hard train rows": r["n_hard_train_rows"],
            "mined": r["n_mined"],
            "mined by rule": r["n_mined_by_rule"] / r["n_hard_train_rows"],
            "mined by fallback": r["n_mined_by_fallback"] / r["n_hard_train_rows"],
            "dropped (no survivor)": r["n_dropped_no_survivor"],
            "drop rate": r["n_dropped_no_survivor"] / r["n_hard_train_rows"],
            "top-1 above threshold": r["n_top1_above_threshold"] / r["n_hard_train_rows"],
            "mined == labeled": r["n_mined_equals_labeled"] / max(r["n_mined"], 1),
            "median rank": r["median_mined_rank"],
        })
    display(pd.DataFrame(rows).round(3))
else:
    print("no mined datasets yet: run mine_hard_negs.py (see paper.sh, negs=mined rows)")

# What the mixer did to the train split, per mixed dataset.
reports = sorted(glob.glob("dataset/processed/*_mixed-*/mixing_report.json"))
if reports:
    rows = []
    for path in reports:
        r = json.load(open(path))
        rows.append({
            "dataset": os.path.basename(os.path.dirname(path)),
            "mined fraction": r["mined_fraction"],
            "eligible train rows": r["n_eligible"],
            "taken from mined": r["n_taken_from_mined"],
            "sources": r["sources"],
        })
    display(pd.DataFrame(rows))

# What the labeling pass measured on the mined negatives, per graded dataset.
reports = sorted(glob.glob("dataset/processed/*_mined-*_graded/labeling_report.json"))
if reports:
    rows = []
    for path in reports:
        r = json.load(open(path))
        rows.append({
            "dataset": os.path.basename(os.path.dirname(path)),
            "mined rows": r["n_mined_rows"],
            "labeled": r["n_labeled"],
            "failed": r["n_failed"],
            "distance 0 (false negative)": r["n_distance_zero"] / r["n_labeled"],
            "clipped at max_distance": r["n_clipped_to_max_distance"] / r["n_labeled"],
            "mean distance": r["mean_distance"],
            "mean fraction violated": r["mean_fraction_violated"],
        })
    display(pd.DataFrame(rows).round(3))


## 7. Full table

Every condition, every cutoff. Also written to paper/figs/all_results.csv.

In [ ]:
table = (results.sort_values(["modality", "query_kind", f"recall@{K_MAIN}"],
                           ascending=[True, True, False])
         .reset_index(drop=True))
table.to_csv(os.path.join(FIG_DIR, "all_results.csv"), index=False)
display(table.round(4))
